# One procedure, every answer we already know

The procedure is short. Measure every stored distance, sort by distance and then by storage index, keep $k$, and vote. Weights, when requested, are $1/d$. A zero distance is refused rather than inverted.

On the six mornings this procedure must reproduce the hand results:

- $k = 1$ busy
- $k = 3$ busy, by $2$ to $1$
- $k = 5$ quiet, by $3$ to $2$
- weighted $k = 5$ busy, by $70/120$ to $59/120$
- the three-neighbor average $20/3$, and the weighted average $336/47$


In [1]:
# The stored mornings, sorted by squared distance, then voted or averaged.
from fractions import Fraction

cases = [
    ("A", 2, 5, "busy", 9),
    ("B", 6, 2, "busy", 8),
    ("C", 5, 6, "quiet", 3),
    ("D", 2, 8, "quiet", 2),
    ("E", 10, 2, "quiet", 1),
    ("F", 7, 14, "busy", 6),
]
query = (2, 2)

def ranked():
    rows = []
    for index, (name, x, y, label, value) in enumerate(cases):
        square = (query[0] - x) ** 2 + (query[1] - y) ** 2
        distance = Fraction(int(square ** 0.5))
        rows.append((square, index, name, distance, label, value))
    rows.sort()
    return rows

order = ranked()
assert [row[2] for row in order] == ["A", "B", "C", "D", "E", "F"]

def vote(k, weighted=False):
    chosen = order[:k]
    totals = {"busy": Fraction(0), "quiet": Fraction(0)}
    for _, _, _, distance, label, _ in chosen:
        totals[label] += Fraction(1, distance) if weighted else 1
    return totals

assert vote(1)["busy"] > vote(1)["quiet"]
assert vote(3) == {"busy": 2, "quiet": 1}
assert vote(5) == {"busy": 2, "quiet": 3}
weighted = vote(5, weighted=True)
assert weighted["busy"] == Fraction(70, 120)
assert weighted["quiet"] == Fraction(59, 120)

def average(k, weighted=False):
    chosen = order[:k]
    if not weighted:
        return Fraction(sum(row[5] for row in chosen), k)
    numerator = sum(Fraction(1, row[3]) * row[5] for row in chosen)
    denominator = sum(Fraction(1, row[3]) for row in chosen)
    return numerator / denominator

assert average(3) == Fraction(20, 3)
assert average(3, weighted=True) == Fraction(336, 47)
print("k=5 weighted", weighted["busy"], weighted["quiet"])
print("regression", average(3), average(3, weighted=True))


k=5 weighted 7/12 59/120
regression 20/3 336/47


## A pitfall

Sorting by distance alone is the bug from the tie lesson. The procedure above sorts by the square and then by the storage index, so a later tie is reproducible. Reproducible is not the same as "there was no tie." If two squares are equal, the report should still say so.

## What to carry forward

One sort produces the order A through F. The votes and the two averages match the fractions already proved.
